# 2교시 · 데이터 조회와 전처리
### — 필요한 것만 남기기

앞 시간에 데이터를 열었고, **빈칸이 있다는 것**을 발견했습니다.
이번 시간에 그 빈칸이 **왜** 비었는지 알아냅니다.

**이 시간이 끝나면 할 수 있는 것**

1. 원하는 행과 열만 골라낼 수 있다
2. 조건에 맞는 데이터만 걸러낼 수 있다
3. 빈칸·중복·형식 오류를 처리할 수 있다
4. **처리 방법을 바꾸면 결론이 달라진다는 것을 안다**

---

> ### 이번 시간의 한 문장
> # 고른다는 것은 버린다는 것이다
>
> 전처리는 데이터를 "깨끗하게 만드는" 작업이 아닙니다.
> **무엇을 버릴지 결정하는 작업**입니다. 그리고 버린 것은 결과에 반드시 영향을 줍니다.

In [ ]:
import pandas as pd

BASE = 'https://raw.githubusercontent.com/JasonWhiteLee/ak-data-analysis-basics/main/'

orders  = pd.read_csv(BASE + 'superstore_orders.csv', parse_dates=['Order Date', 'Ship Date'])
returns = pd.read_csv(BASE + 'superstore_returns.csv')

print(orders.shape)
orders.head(3)

---
# 2-1. 골라내기 — 열 고르기

## 열 하나 / 여러 개

앞 시간에 했던 것입니다. 대괄호 하나면 **Series**, 두 겹이면 **DataFrame**.

In [ ]:
orders['Sales'].head(3)

In [ ]:
orders[['Order Date', 'Category', 'Sales']].head(3)

## 필요 없는 열 버리기 — `.drop()`

`axis=1` 은 "열 방향"이라는 뜻입니다 (`axis=0` 은 행).

In [ ]:
# Row ID 는 그냥 번호라 분석에 쓸 일이 없습니다
작은표 = orders.drop(['Row ID'], axis=1)

print('원래:', orders.shape)
print('버린뒤:', 작은표.shape)

> ### 잠깐 — 이 열은 왜 남겨 둘까요
>
> 1교시에 `Country/Region` 을 봤습니다. 미국 10,038 / 캐나다 201 이었죠.
> 거의 다 미국인데, **완전히 하나는 아닙니다.**
>
> 이런 열은 버릴까요 남길까요? — **답은 "무엇을 하려는가"에 달려 있습니다.**
> 국내 매출만 볼 거면 캐나다를 빼야 하고, 전체를 볼 거면 남겨야 합니다.
>
> **버리기 전에 무엇을 버리는지 확인하는 것.** 이게 전처리의 전부입니다.

---
# 2-2. loc 와 iloc — 위치로 고르기

pandas에는 행·열을 고르는 전용 도구가 둘 있습니다. 이름이 비슷해서 늘 헷갈립니다.

| | 무엇으로 고르나 | 예 |
|---|---|---|
| **`.loc`** | **이름(라벨)** 으로 | `orders.loc[0, 'Sales']` |
| **`.iloc`** | **순서(번호)** 로 | `orders.iloc[0, 17]` |

`i` 는 **index(번호)** 의 i 라고 외우면 됩니다.

In [ ]:
# loc — [행, 열이름]
orders.loc[0, 'Sales']

In [ ]:
# iloc — [행번호, 열번호]
orders.iloc[0, 0]

## 범위로 고르기

`:` 는 "전부" 또는 "부터 까지"라는 뜻입니다.

In [ ]:
# 0~2행, Order Date 부터 Segment 까지
orders.loc[0:2, 'Order Date':'Segment']

In [ ]:
# 앞 3행, 앞 5열
orders.iloc[0:3, 0:5]

> ### 하나 다른 점
> `.loc[0:2]` 는 **2번 행을 포함**하고, `.iloc[0:2]` 는 **2번 행을 포함하지 않습니다.**
> 이름으로 고를 땐 끝을 포함하고, 번호로 자를 땐 포함하지 않습니다.
>
> 헷갈리면 **결과 행 수를 세어 보면 됩니다.** 오늘은 `.loc` 만 주로 씁니다.

---
# 2-3. 걸러내기 — 조건에 맞는 행만

**엑셀의 필터**와 같습니다. 오늘 가장 많이 쓸 기능입니다.

## 조건 하나

먼저 조건만 써 보면, `True` / `False` 가 줄줄이 나옵니다.

In [ ]:
(orders['Sales'] > 1000).head()

이 참·거짓 목록을 대괄호에 넣으면, **참인 행만** 남습니다.

In [ ]:
큰주문 = orders[orders['Sales'] > 1000]

print(len(큰주문), '건')
큰주문[['Order Date', 'Category', 'Sales']].head()

## 조건 두 개 이상

- `&` = 그리고 (and)
- `|` = 또는 (or)
- **각 조건을 반드시 괄호로 감쌉니다.** 안 감싸면 에러가 납니다.

In [ ]:
조건 = (orders['Category'] == 'Technology') & (orders['Sales'] > 1000)

기술고액 = orders[조건]
print(len(기술고액), '건')
기술고액[['Category', 'Sales', 'Profit']].head()

## 자주 쓰는 조건 세 가지

In [ ]:
# 여러 값 중 하나 — isin
서부동부 = orders[orders['Region'].____]
print('isin      :', len(서부동부))

# 사이 값 — between
중간가격 = orders[orders['Sales'].between(100, 500)]
print('between   :', len(중간가격))

# 글자 포함 — str.contains
의자 = orders[orders['Product Name'].str.contains('Chair', case=False)]
print('contains  :', len(의자))

## 날짜로 거르기

1교시에 `parse_dates` 로 날짜를 **진짜 날짜**로 읽었기 때문에 이런 게 됩니다.

In [ ]:
작년 = orders[orders['Order Date'] >= '2026-01-01']

print(len(작년), '건')
print(작년['Order Date'].min(), '~', 작년['Order Date'].max())

> **만약 `parse_dates` 를 안 했다면** 이 비교는 글자끼리 비교가 되어
> 엉뚱한 결과가 나오거나 에러가 납니다. 1교시 1-2의 "값의 종류"가 여기서 돌아옵니다.

## 정렬 — `.sort_values()`

In [ ]:
orders.sort_values('Sales', ascending=False)[['Product Name', 'Sales', 'Profit']].head(5)

> 가장 큰 주문의 `Sales` 를 보세요. 다음 시간에 다시 만납니다.

---
# 2-4. 빈칸 — 1교시의 그 질문

1교시 마지막에 `Ship Date` 에 빈칸이 있는 걸 발견했습니다.
**왜 비었을까요?**

보통은 여기서 바로 `fillna()` 나 `dropna()` 를 씁니다. **그게 사고의 시작입니다.**

In [ ]:
orders.isna().sum()

## 먼저 물어야 할 것 — "이 빈칸은 랜덤인가"

빈칸이 **아무데나 흩어져 있으면** 지워도 큰 문제가 없습니다.
하지만 **특정 종류의 행에 몰려 있으면**, 지우는 순간 그 종류가 통째로 사라집니다.

확인해 봅시다. 반품 목록을 붙여서 비교합니다.

In [ ]:
반품주문 = set(returns['Order ID'])

orders['반품여부']  = orders['Order ID'].isin(반품주문)
orders['배송일없음'] = orders['Ship Date'].isna()

pd.crosstab(orders['반품여부'], orders['배송일없음'])

In [ ]:
print('반품 건 중 배송일이 빈 비율  : {:.1f}%'.format(
      orders[orders['반품여부']]['배송일없음'].mean() * 100))
print('반품 아닌 건 중              : {:.1f}%'.format(
      orders[~orders['반품여부']]['배송일없음'].mean() * 100))

## 결과를 보세요

**반품 건은 35%가 비어 있고, 반품이 아닌 건은 0.4%뿐입니다.**

빈칸이 랜덤이 아닙니다. **반품된 주문에 몰려 있습니다.**
현실을 상상해 보면 이해가 됩니다 — 반품되면 배송 기록이 제대로 안 남는 겁니다.

> ### 그래서 이 빈칸은 "없는 값"이 아닙니다
> **"반품됐다"는 정보 그 자체입니다.**
>
> 앞 시간의 이야기가 여기서 돌아옵니다 —
> **기록되지 않은 것이 오히려 많은 것을 말해 줍니다.**

## 그냥 지우면 어떻게 되는지 직접 봅시다

In [ ]:
def 반품률(표):
    주문별 = 표.groupby('Order ID')['반품여부'].first()
    return 주문별.mean() * 100

지운표 = orders.dropna(subset=['Ship Date'])

print('원본        : 반품률 {:.2f}%   (주문 {}건)'.format(반품률(orders), orders['Order ID'].nunique()))
print('dropna 후   : 반품률 {:.2f}%   (주문 {}건)'.format(반품률(지운표), 지운표['Order ID'].nunique()))

## 반품률이 5.79% 에서 5.19% 로 떨어졌습니다

**아무도 거짓말을 하지 않았습니다.** 빈칸을 지웠을 뿐입니다.
그런데 반품률이 **10% 축소**되어 보고됩니다.

> ### 이번 시간의 핵심
> `dropna()` 는 "정리"가 아니라 **판단**입니다.
> 그리고 그 판단은 **결과 숫자를 바꿉니다.**
>
> 코드는 한 줄이지만, 그 한 줄이 보고서의 결론을 바꿉니다.
> **이것이 사람이 해야 하는 일입니다.** 도구는 시키는 대로 지울 뿐입니다.

## 처리 방법 세 가지

| 방법 | 코드 | 언제 |
|---|---|---|
| **버린다** | `df.dropna(subset=['열'])` | 빈칸이 랜덤이고, 양이 적을 때 |
| **채운다** | `df['열'].fillna(값)` | 빈칸의 의미를 알고, 대체값이 타당할 때 |
| **표시한다** | `df['열'].isna()` 로 새 열 | **빈칸 자체가 정보일 때** |

이 데이터에서는 세 번째가 맞습니다. 이미 `배송일없음` 열을 만들어 뒀죠.

In [ ]:
# 채우는 예 — 빈 우편번호를 '미상' 으로
orders['Postal Code'].fillna('미상').head(3)

> `fillna(0)` 을 습관적으로 쓰는 것이 가장 위험합니다.
> **비어 있는 것과 0은 다릅니다.** 매출이 "없는 것"과 "0원인 것"은 완전히 다른 이야기입니다.

---
# 2-5. 중복 — 같은 줄이 두 번 들어왔을 때

시스템에서 두 번 내려받거나, 파일을 잘못 합치면 생깁니다.

In [ ]:
print('완전히 똑같은 행:', orders.duplicated().____)

In [ ]:
# 실제로 어떤 행인지 보기
orders[orders.duplicated(keep=False)].sort_values('Row ID').head(4)

In [ ]:
중복제거 = orders.drop_duplicates()

print('전:', len(orders))
print('후:', len(중복제거))

> ### 주의 — 진짜 중복인지 확인하고 지웁니다
>
> 1교시에서 봤듯이 **한 주문에 여러 품목**이 있으면 `Order ID` 가 반복됩니다.
> 이건 중복이 아니라 정상입니다.
>
> `drop_duplicates()` 는 **모든 열이 똑같은 행**만 지웁니다.
> 특정 열 기준으로 지우려면 `subset=` 을 쓰는데, **이때 진짜 데이터가 날아갑니다.**
>
> ```python
> df.drop_duplicates(subset=['Order ID'])   # 주문당 1줄만 남김 = 품목 정보 소멸
> ```

---
# 2-6. 형식 바꾸기 — 계산이 안 되는 대부분의 원인

1교시 1-2에서 `'1500' + '900'` 이 `'1500900'` 이 되는 걸 봤습니다.
실제 데이터에서 이 문제가 어떻게 나타나는지 봅니다.

In [ ]:
orders.dtypes

## 숫자로 바꾸기 — `.astype()` / `pd.to_numeric()`

In [ ]:
# 예시 — 숫자처럼 생겼지만 글자인 열
샘플 = pd.DataFrame({'금액': ['1500', '900', '700']})
print(샘플.dtypes)
print('더하면:', 샘플['금액'].sum())      # 이어붙습니다

In [ ]:
샘플['금액'] = 샘플['금액'].astype(int)

print(샘플.dtypes)
print('더하면:', 샘플['금액'].sum())      # 이제 계산됩니다

> **`errors='coerce'` 를 기억하세요.**
> 숫자로 못 바꾸는 값(예: `'미상'`)이 섞여 있으면 `astype` 은 에러를 냅니다.
> `pd.to_numeric(열, errors='coerce')` 를 쓰면 **못 바꾸는 값을 빈칸으로** 만들고 넘어갑니다.

## 날짜에서 조각 꺼내기 — `.dt`

날짜 열에 `.dt` 를 붙이면 연·월·요일을 꺼낼 수 있습니다.
**월별 집계를 하려면 반드시 필요합니다.** (다음 시간에 씁니다)

In [ ]:
orders['연도']  = orders['Order Date'].dt.____
orders['월']    = orders['Order Date'].dt.month
orders['요일']  = orders['Order Date'].dt.day_name()

orders[['Order Date', '연도', '월', '요일']].head()

## 파생 열 만들기 — 없는 정보를 계산해 낸다

두 날짜의 차이로 **배송에 걸린 날짜**를 구할 수 있습니다.

In [ ]:
orders['배송일수'] = (orders['Ship Date'] - orders['Order Date']).dt.days

orders[['Order Date', 'Ship Date', '배송일수']].head()

In [ ]:
orders['배송일수'].describe()

> `count` 를 보세요. 전체 행 수보다 적습니다.
> **`Ship Date` 가 빈 행은 배송일수도 자동으로 비어 있습니다.**
> 빈칸은 이렇게 조용히 아래로 퍼져 나갑니다.

---
# 2-7. 이제 진짜 지저분한 데이터

지금까지 쓴 데이터는 정리된 편입니다.
**실무에서 받는 데이터는 이렇지 않습니다.**

영국의 어느 온라인 소매점 **1년치 거래 기록 54만 건**을 열어 봅니다.

In [ ]:
retail = pd.read_csv(BASE + 'online_retail.csv', parse_dates=['InvoiceDate'])

print(retail.shape)
retail.head()

In [ ]:
retail.info()

## 문제 1 — 고객 번호의 25%가 없습니다

In [ ]:
print('CustomerID 결측: {:,}건 ({:.1f}%)'.format(
      retail['CustomerID'].isna().sum(), retail['CustomerID'].isna().mean() * 100))

54만 건 중 13만 건에 고객 번호가 없습니다.

**이걸 지우면 매출의 4분의 1이 사라집니다.**
하지만 남기면 "고객당 구매액" 같은 분석을 할 수 없습니다.

> **정답이 없습니다.** 무엇을 하려는지에 따라 다릅니다.
> - 고객 분석 -> 지워야 함. 대신 **"비회원 제외"라고 보고서에 적어야 합니다**
> - 매출 집계 -> 남겨야 함
>
> 중요한 건 **어느 쪽을 골랐는지 밝히는 것**입니다.

## 문제 2 — 취소 주문이 섞여 있습니다

이 데이터는 취소 건의 송장번호가 **`C` 로 시작**합니다.

In [ ]:
retail['취소'] = retail['InvoiceNo'].astype(str).str.startswith('C')

print('취소 건수:', retail['취소'].sum())
retail[retail['취소']].head(3)

`Quantity` 가 **음수**인 것을 보세요. 취소는 수량을 빼는 방식으로 기록돼 있습니다.

**모르고 매출을 합치면 어떻게 될까요?**

In [ ]:
retail['금액'] = retail['Quantity'] * retail['UnitPrice']

print('그대로 합계    : {:>12,.0f}'.format(retail['금액'].sum()))
print('취소 제외 합계 : {:>12,.0f}'.format(retail[~retail['취소']]['금액'].sum()))

**90만 가까이 차이납니다.**

재미있는 건 **어느 쪽도 틀리지 않았다**는 점입니다.
- "실제로 들어온 돈"을 알고 싶으면 -> 취소를 포함한 순매출
- "얼마나 팔렸는지"를 알고 싶으면 -> 취소 제외한 총매출

> **숫자를 내기 전에 무엇을 알고 싶은지 정해야 합니다.**
> 순서가 반대면, 나온 숫자에 맞춰 질문을 만들게 됩니다.

## 문제 3 — 단가가 0이거나 음수입니다

In [ ]:
print('단가 0 이하:', (retail['UnitPrice'] <= 0).sum(), '건')

retail[retail['UnitPrice'] <= 0]['Description'].value_counts().head(5)

설명을 보면 정체가 드러납니다 — 사은품, 재고 조정, 파손 처리 같은 것들입니다.

**이건 오류가 아니라 다른 종류의 기록입니다.**
매출 분석에서는 빼야 하지만, "왜 이렇게 많이 파손됐나"를 볼 거라면 이게 본체입니다.

## 문제 4 — 완전히 똑같은 행이 5천 개

In [ ]:
print('완전중복:', retail.duplicated().sum(), '건')

## 문제 5 — 이건 어느 나라 데이터인가

In [ ]:
retail['Country'].value_counts().head(5)

38개국이 있지만 **영국이 91%** 입니다.

> "글로벌 온라인 소매 데이터"라고 부르면 틀린 말이 됩니다.
> **1교시의 질문이 여기서 다시 나옵니다 — 이 데이터는 누구를 대표합니까?**

---
# 2-8. 실습 — 내가 버린 것을 밝히기

아래 순서로 `retail` 을 정리하고, **매 단계마다 몇 행이 사라졌는지** 기록하세요.

In [ ]:
print('시작            : {:>7,} 행'.format(len(retail)))

단계1 = retail.drop_duplicates()
print('중복 제거 후    : {:>7,} 행   (-{:,})'.format(len(단계1), len(retail) - len(단계1)))

단계2 = 단계1[~단계1['취소']]
print('취소 제외 후    : {:>7,} 행   (-{:,})'.format(len(단계2), len(단계1) - len(단계2)))

단계3 = 단계2[단계2['UnitPrice'] > 0]
print('단가 0 제외 후  : {:>7,} 행   (-{:,})'.format(len(단계3), len(단계2) - len(단계3)))

단계4 = 단계3.dropna(subset=['CustomerID'])
print('비회원 제외 후  : {:>7,} 행   (-{:,})'.format(len(단계4), len(단계3) - len(단계4)))

print()
print('전체의 {:.1f}% 를 버렸습니다'.format((1 - len(단계4) / len(retail)) * 100))

## 여기에 적으세요

**이 텍스트 셀을 더블클릭**해서 채우세요.

### 내가 버린 것 (각각 몇 %, 그리고 왜)
1. 중복 —
2. 취소 —
3. 단가 0 —
4. 비회원 —

### 이 정리된 데이터로 **답할 수 없게 된** 질문 하나


### 만약 "비회원 제외"를 하지 않았다면, 어떤 결론이 달라질까요


---

> ### 마지막 질문
> 위 네 단계 중 **하나를 잘못 판단하면** 보고서의 어느 숫자가 바뀝니까?
>
> 앞에서 본 것처럼 — `dropna()` 한 줄로 반품률이 5.79%에서 5.19%가 됐습니다.
> **버린 것을 밝히지 않은 보고서는, 틀린 게 아니라 검증할 수 없는 보고서입니다.**

---
# 정리 — 오늘 쓴 것

## 코드

| 하는 일 | 코드 |
|---|---|
| 열 고르기 | `df['열']` · `df[['열1','열2']]` |
| 열 버리기 | `df.drop(['열'], axis=1)` |
| 이름으로 고르기 | `df.loc[행, '열']` |
| 번호로 고르기 | `df.iloc[행번호, 열번호]` |
| 조건 필터 | `df[df['열'] > 100]` |
| 조건 여러 개 | `df[(조건1) & (조건2)]` |
| 여러 값 중 하나 | `df['열'].isin([...])` |
| 사이 값 | `df['열'].between(a, b)` |
| 글자 포함 | `df['열'].str.contains('...')` |
| 정렬 | `df.sort_values('열', ascending=False)` |
| 빈칸 세기 | `df.isna().sum()` |
| 빈칸 버리기 | `df.dropna(subset=['열'])` |
| 빈칸 채우기 | `df['열'].fillna(값)` |
| 중복 확인 / 제거 | `df.duplicated().sum()` · `df.drop_duplicates()` |
| 형 바꾸기 | `df['열'].astype(int)` · `pd.to_numeric(..., errors='coerce')` |
| 날짜 조각 | `df['날짜'].dt.year` · `.dt.month` · `.dt.day_name()` |
| 교차표 | `pd.crosstab(A, B)` |

## 남길 것 세 가지

1. **빈칸을 지우기 전에 "왜 비었는지" 묻는다** — 반품 건에 몰려 있었습니다
2. **버린 것은 결과를 바꾼다** — `dropna()` 한 줄로 반품률이 10% 줄었습니다
3. **무엇을 버렸는지 밝힌다** — 밝히지 않은 보고서는 검증할 수 없습니다

---

### 다음 시간

정리한 데이터를 **묶어서 요약하고, 다른 표와 이어 붙입니다.**
그리고 이런 질문을 하게 됩니다 — **"매출 3억"은 좋은 겁니까, 나쁜 겁니까?**